# Token embeddings $\times$ contextual embeddings $\times$ sentence embeddings.

## Modelos

In [ ]:
%pip install -q transformers sentence-transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

# LLM causal: prevê o próximo token
nome_llm = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(nome_llm)
llm = AutoModelForCausalLM.from_pretrained(nome_llm).to(device)
llm.eval()

# Modelo de embeddings semânticos
nome_st = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st = SentenceTransformer(nome_st, device=device)

Dispositivo: cuda


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Tokens embeddings, embeddings iniciais: (1, 6, 896)

In [36]:
texto = "Data visualization empowers users to"

inputs = tokenizer(texto, return_tensors="pt").to(device)
ids = inputs["input_ids"][0]

# Converter os IDs em representações textuais dos tokens
tokens = tokenizer.convert_ids_to_tokens(ids.tolist())

# Camada inicial de embeddings: consulta à matriz embed_tokens
with torch.no_grad():
    embeddings_iniciais = llm.model.embed_tokens(inputs["input_ids"])

print("Texto:", texto)
print("\nTokens e IDs:")

for token, token_id in zip(tokens, ids.tolist()):
    print(f"{token!r:20} ID={token_id}")

print("\nFormato dos IDs:", tuple(inputs["input_ids"].shape))
print("Formato dos embeddings iniciais:",
      tuple(embeddings_iniciais.shape))

Texto: Data visualization empowers users to

Tokens e IDs:
'Data'               ID=1043
'Ġvisualization'     ID=41048
'Ġemp'               ID=8486
'owers'              ID=15965
'Ġusers'             ID=3847
'Ġto'                ID=311

Formato dos IDs: (1, 6)
Formato dos embeddings iniciais: (1, 6, 896)


## Contextual embeddings: saída do transformers

In [37]:
with torch.no_grad():
    saida = llm(
        **inputs,
        output_hidden_states=True
    )

# hidden_states[0]: embeddings de entrada, antes das camadas
# hidden_states[-1]: saída da última camada Transformer
contextualizados = saida.hidden_states[-1]

print("Embeddings iniciais:",
      tuple(saida.hidden_states[0].shape))

print("Representações contextualizadas:",
      tuple(contextualizados.shape))

# Comparar os vetores correspondentes ao último token
v_inicial = saida.hidden_states[0][0, -1]
v_contextual = contextualizados[0, -1]

print("\nNorma do embedding inicial:",
      v_inicial.norm().item())

print("Norma da representação contextualizada:",
      v_contextual.norm().item())

print("Similaridade de cosseno entre os dois:",
      torch.nn.functional.cosine_similarity(
          v_inicial.unsqueeze(0),
          v_contextual.unsqueeze(0)
      ).item())

Embeddings iniciais: (1, 6, 896)
Representações contextualizadas: (1, 6, 896)

Norma do embedding inicial: 0.423828125
Norma da representação contextualizada: 256.0
Similaridade de cosseno entre os dois: 0.0947265625


## Candidatos do Próximo Token

In [38]:
with torch.no_grad():
    logits = llm(**inputs).logits[0, -1, :]

probabilidades = torch.softmax(logits.float(), dim=-1)
top = torch.topk(probabilidades, k=10)

print("Possíveis próximos tokens:\n")

for token_id, prob in zip(
    top.indices.tolist(),
    top.values.tolist()
):
    token_texto = tokenizer.decode([token_id])
    print(f"{token_texto!r:20} {prob:.3%}")

Possíveis próximos tokens:

' explore'           15.102%
' analyze'           8.083%
' understand'        8.083%
' gain'              6.295%
' make'              5.556%
' better'            3.818%
' quickly'           3.370%
' interpret'         3.370%
' visualize'         2.974%
' engage'            2.044%


## Gerando a frase

In [39]:
with torch.no_grad():
    gerado = llm.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

# Separar o prompt original dos tokens novos
novos_ids = gerado[0, inputs["input_ids"].shape[1]:]
continuacao = tokenizer.decode(
    novos_ids,
    skip_special_tokens=True
)

print("Prompt:", texto)
print("Continuação gerada:", repr(continuacao))
print("Frase completa:", texto + continuacao)

Prompt: Data visualization empowers users to
Continuação gerada: ' explore and understand complex data sets in a more intuitive and engaging way. It allows users to visualize data'
Frase completa: Data visualization empowers users to explore and understand complex data sets in a more intuitive and engaging way. It allows users to visualize data


## Sentence embeddings

In [40]:
frases = [
    "Data visualization empowers users to explore complex data sets.",
    "To be or not to be, that is the question.",
    "Data science can be used to answer many questions."
]

vetores = st.encode(
    frases,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Formato dos embeddings:", tuple(vetores.shape))

for frase, vetor in zip(frases, vetores):
    print(f"\nFrase: {frase}")
    print("Dimensão:", vetor.numel())
    print("Primeiros 8 valores:", vetor[:8].cpu().tolist())

Formato dos embeddings: (3, 384)

Frase: Data visualization empowers users to explore complex data sets.
Dimensão: 384
Primeiros 8 valores: [-0.0340808667242527, -0.018799610435962677, -0.016429414972662926, -0.03187555819749832, -0.04344017803668976, -0.02867666631937027, 0.037536073476076126, -0.0410224087536335]

Frase: To be or not to be, that is the question.
Dimensão: 384
Primeiros 8 valores: [0.17177996039390564, 0.05468443036079407, 0.029985500499606133, 0.03953665494918823, 0.0025653941556811333, -0.033963754773139954, 0.03110133484005928, -0.026221077889204025]

Frase: Data science can be used to answer many questions.
Dimensão: 384
Primeiros 8 valores: [0.02058914676308632, 0.024534102529287338, -0.05796224623918533, -0.0026734010316431522, -0.00737283518537879, -0.0365619957447052, -0.05632184445858002, 0.04608999565243721]


In [41]:
similaridades = vetores @ vetores.T

print("Similaridade de cosseno:\n")
print(similaridades.cpu().numpy().round(3))

Similaridade de cosseno:

[[ 1.    -0.058  0.555]
 [-0.058  1.     0.186]
 [ 0.555  0.186  1.   ]]
